### Set up the Critique Agent

In [ ]:
import os
from openai import AzureOpenAI

deployment = "gpt-4.1"

client = AzureOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint="https://ai-eastus2279869221189.openai.azure.com/",
    api_key="",
)

### Set up the experiment

In [ ]:
import datetime
import os
experiment_name = datetime.datetime.now().strftime('experiment_%Y%m%d%H%M%S')

os.makedirs( experiment_name, exist_ok=True )

iterations = set()

experiment_name

## Set the Initial Prompt

In [ ]:




object_to_generate = "table"

obj_prompt = f"Create a 3D model of a {object_to_generate} in OBJ format."



# ......... 1-N Loop ...............

## Echo updated 3D Agent generation prompt

In [ ]:
import urllib.request
import json

iterations.add(obj_prompt)


# Request data goes here
# The example below assumes JSON formatting which may be updated
# depending on the format your endpoint expects.
# More information can be found here:
# https://docs.microsoft.com/azure/machine-learning/how-to-deploy-advanced-entry-script
data = {
  "inputs": obj_prompt,
  "parameters": {
    "do_sample": True,
    "top_p": 0.95,
    "temperature": 0.2,
    "top_k": 50,
    "repetition_penalty": 1.03,
    "stop": ["\nUser:", "<|endoftext|>", "</s>"]
  }
}

body = str.encode(json.dumps(data))

url = 'https://gpu-workspace-uksouth-mepds.uksouth.inference.ml.azure.com/generate'
# Replace this with the primary/secondary key, AMLToken, or Microsoft Entra ID token for the endpoint
api_key = ''
if not api_key:
    raise Exception("A key should be provided to invoke the endpoint")


headers = {'Content-Type':'application/json', 'Accept': 'application/json', 'Authorization':('Bearer '+ api_key)}

req = urllib.request.Request(url, body, headers)

try:
    response = urllib.request.urlopen(req)

    result = response.read()
    print(result)
    result_json = json.loads(result)

    generated_data = result_json["generated_text"]
    print(generated_data)

    with open(f'{experiment_name}/{object_to_generate}_{len(iterations)}.obj', 'w') as f:
        obj_data = generated_data.split("```obj")[1].split("```")[0].strip()
        f.write(obj_data)

    print(result_json["generated_text"])

    print(f"3D model saved to {object_to_generate}.obj")
except urllib.error.HTTPError as error:
    print("The request failed with status code: " + str(error.code))

    # Print the headers - they include the requert ID and the timestamp, which are useful for debugging the failure
    print(error.info())
    print(error.read().decode("utf8", 'ignore'))

## Generate Critique

In [ ]:
response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": f"""The following .OBJ 3D model was generate by another AI Assistant.  
            It is supposed to be a {object_to_generate}.
            Analyze the and generated model and determine if it is indeed a {object_to_generate}.
            Provide clear and concise instructions to the 3D Generator AI to improve the {object_to_generate}.

            Instruction length should be less than 100 words.

            Do not include the analysis in the response.  Only provide the instructions.
            
            {obj_data}
            """,
        }
    ],
    temperature=1.0,
    top_p=1.0,
    frequency_penalty=0.0,
    presence_penalty=0.0,
    model=deployment
)

critique = response.choices[0].message.content
print("Critique and Improvement Instructions:") 

print(critique)

In [ ]:
obj_prompt = f"""Modify the following 3D model: 

{obj_data}

according to these instructions:

{critique}

Respond with a new 3D model in OBJ format.
"""

In [ ]:
!pip install pywavefront

In [ ]:
import pywavefront
# Load the OBJ file
scene = pywavefront.Wavefront("experiment_20250902111334/table_1.obj", collect_faces=True)
# Access vertices and faces
print("Vertices:", scene.vertices)
for mesh in scene.mesh_list:
   print("Faces:", mesh.faces)

In [ ]:
!pip install PyOpenGL PyWavefront

In [ ]:
!pip install -U pyglet==1.5.21

In [ ]:
import pywavefront
from pywavefront.visualization import draw
import pyglet

# Load the .obj file
scene = pywavefront.Wavefront("experiment_20250902111334/table_1.obj", collect_faces=True)

# Create a Pyglet window
window = pyglet.window.Window(width=800, height=600, caption="OBJ Renderer")

@window.event
def on_draw():
    window.clear()
    draw(scene)  # Render the .obj file

# Run the Pyglet application
pyglet.app.run()


In [ ]:
import trimesh
# Load the OBJ file
mesh = trimesh.load("experiment_20250902102039/table_1.obj")
# Print basic information
print(mesh)
# Visualize the model
mesh.show()

In [ ]:
import numpy as np

# Create a scene and add the mesh
scene = trimesh.Scene()
scene.add_geometry(mesh, node_name='my_mesh')

# Define a transformation matrix (e.g., translation)
translation = np.eye(4)  # Identity matrix
translation[:3, 3] = [1.0, 2.0, 3.0]  # Move by (x=1.0, y=2.0, z=3.0)

# Apply the transformation to the object in the scene
scene.graph.update(frame_from='my_mesh', matrix=translation)

# Visualize the scene
scene.show()

In [8]:
#ffmpeg.exe -framerate 1 -i "image%010d.png" -c:v libx264 -r 30 -pix_fmt yuv420p ..\flyby1.mp4